# 03 - Preprocessing, Normalization and Splitting

**Phase 1, step 3.** Turn the cleaned metadata into model-ready arrays.

| Step | Decision | Reason (from EDA) |
|---|---|---|
| Grayscale | 1 channel | Eye state does not need colour; infrared sensors are grayscale anyway |
| Resize | 64 x 64 | Image sizes vary; CNNs need one fixed size |
| Scaling | pixels / 255 -> [0, 1] | Pixels span 0-255; small inputs make training stable |
| Split | subject-wise 70 / 15 / 15 | Images of the same person look alike; mixing them between train and test would leak information |
| Class weights | computed from the training set | Only needed if the classes are imbalanced |

Outputs (in `data/processed/`): `eye_dataset_64.npz`, `splits.csv`, `normalization_stats.json`.

In [ ]:
import os, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import GroupShuffleSplit

sns.set_theme(style='whitegrid')
SEED = 42
IMG = 64
MAX_IMAGES = None       # set e.g. 20000 for a quick test run; None = use everything
DATA_DIR = os.environ.get('MRL_DIR', '../data/raw/mrl')
PROC_DIR, IMG_DIR = '../data/processed', '../images'
os.makedirs(IMG_DIR, exist_ok=True)
def savefig(name): plt.savefig(os.path.join(IMG_DIR, name), dpi=150, bbox_inches='tight')

df = pd.read_csv(f'{PROC_DIR}/metadata_clean.csv')
if MAX_IMAGES and len(df) > MAX_IMAGES:
    df = df.sample(MAX_IMAGES, random_state=SEED).reset_index(drop=True)
print(len(df), 'images,', df.subject.nunique(), 'subjects')

## 1. Subject-wise train / validation / test split
`GroupShuffleSplit` keeps all images of one person in the same split. We assert that no subject appears in two splits.

In [ ]:
groups = df.subject.values
tr_idx, rest_idx = next(GroupShuffleSplit(1, test_size=0.30, random_state=SEED).split(df, df.closed, groups))
rest = df.iloc[rest_idx]
va_rel, te_rel = next(GroupShuffleSplit(1, test_size=0.50, random_state=SEED).split(rest, rest.closed, rest.subject.values))
va_idx, te_idx = rest_idx[va_rel], rest_idx[te_rel]

df['split'] = ''
df.loc[tr_idx, 'split'] = 'train'; df.loc[va_idx, 'split'] = 'val'; df.loc[te_idx, 'split'] = 'test'

s = {k: set(df[df.split == k].subject) for k in ['train', 'val', 'test']}
assert not (s['train'] & s['val']) and not (s['train'] & s['test']) and not (s['val'] & s['test']), 'subject leakage!'
assert (df.split != '').all()

summary = df.groupby('split').agg(images=('closed', 'size'), subjects=('subject', 'nunique'), closed_share=('closed', 'mean')).round(3)
print(summary.loc[['train', 'val', 'test']].to_string())

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
summary.loc[['train', 'val', 'test'], 'images'].plot.bar(ax=ax[0], color='steelblue', rot=0); ax[0].set_title('Images per split')
(summary.loc[['train', 'val', 'test'], 'closed_share'] * 100).plot.bar(ax=ax[1], color='indianred', rot=0)
ax[1].axhline(50, color='k', ls='--', lw=.8); ax[1].set_title('% closed eyes per split'); ax[1].set_ylabel('%')
plt.tight_layout(); savefig('prep_01_split_summary.png'); plt.show()

**What it tells us:** no subject is shared between the splits (checked by the assertion), and the share of closed eyes is similar across train / val / test, so the splits are comparable.

## 2. Load images: grayscale + resize
Images are stored as `uint8` (0-255) to keep the file small. Scaling to [0, 1] is applied in the next section.

In [ ]:
def load_gray(rel):
    with Image.open(os.path.join(DATA_DIR, rel)) as im:
        return np.asarray(im.convert('L').resize((IMG, IMG), Image.BILINEAR), dtype=np.uint8)

data = {}
for split in ['train', 'val', 'test']:
    sub = df[df.split == split]
    X = np.stack([load_gray(r) for r in tqdm(sub.rel_path, desc=split)])[..., None]   # (N, 64, 64, 1)
    data[split] = (X, sub.closed.values.astype(np.int8))
    print(split, X.shape, X.dtype, 'min/max:', X.min(), X.max())

## 3. Normalization / scaling
Pixels are scaled from `[0, 255]` to `[0, 1]` by dividing by 255. This is a fixed, label-independent operation, so there is no leakage. We also compute the pixel mean and standard deviation **from the training set only**, in case standardisation is wanted in Phase 2.

In [ ]:
def scale(X):                      # uint8 [0,255] -> float32 [0,1]
    return X.astype(np.float32) / 255.0

Xtr_s = scale(data['train'][0])
stats = {'img_size': IMG, 'scale': 'pixel / 255.0', 'train_pixel_mean': float(Xtr_s.mean()), 'train_pixel_std': float(Xtr_s.std()),
         'label_mapping': {'0': 'open', '1': 'closed'}, 'seed': SEED}
print(json.dumps(stats, indent=2))

# verification
assert Xtr_s.min() >= 0.0 and Xtr_s.max() <= 1.0
Xstd = (Xtr_s - stats['train_pixel_mean']) / stats['train_pixel_std']
print(f'Scaled range: [{Xtr_s.min():.2f}, {Xtr_s.max():.2f}] | standardised mean {Xstd.mean():.3f}, std {Xstd.std():.3f}')

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].hist(data['train'][0].ravel()[::20], bins=50, color='gray'); ax[0].set_title('Before: pixel values 0-255')
ax[1].hist(Xtr_s.ravel()[::20], bins=50, color='steelblue'); ax[1].set_title('After scaling: 0-1')
ax[2].hist(Xstd.ravel()[::20], bins=50, color='seagreen'); ax[2].set_title('Optional standardisation (mean 0, std 1)')
plt.tight_layout(); savefig('prep_02_before_after_scaling.png'); plt.show()

**What it tells us:** the shape of the histogram is unchanged, only the range is. Scaling to [0, 1] puts all inputs on a small, common scale, which helps gradient-based training converge.

## 4. Class weights (from the training set only)

In [ ]:
ytr = data['train'][1]
cnt = np.bincount(ytr, minlength=2)
class_weights = {0: len(ytr) / (2 * cnt[0]), 1: len(ytr) / (2 * cnt[1])}
stats['class_weights'] = {str(k): float(v) for k, v in class_weights.items()}
print('Training counts (open, closed):', cnt.tolist())
print('Suggested class weights:', {k: round(v, 3) for k, v in class_weights.items()})

## 5. Visual check of the processed images

In [ ]:
X, y = data['train']
rng = np.random.default_rng(SEED)
fig, ax = plt.subplots(2, 8, figsize=(16, 4.5))
for r, lab in enumerate([0, 1]):
    idx = rng.choice(np.where(y == lab)[0], 8, replace=False)
    for c, i in enumerate(idx):
        ax[r, c].imshow(scale(X[i:i+1])[0, ..., 0], cmap='gray', vmin=0, vmax=1); ax[r, c].axis('off')
    ax[r, 0].set_title('open' if lab == 0 else 'closed', loc='left')
plt.suptitle(f'Processed training images ({IMG}x{IMG}, grayscale, scaled to [0,1])')
plt.tight_layout(); savefig('prep_03_processed_samples.png'); plt.show()

## 6. Save the processed dataset
`eye_dataset_64.npz` holds `X_train, y_train, X_val, y_val, X_test, y_test` (images as `uint8`; apply `/255` when loading, see the helper below). `splits.csv` lists which file belongs to which split.

In [ ]:
np.savez_compressed(f'{PROC_DIR}/eye_dataset_64.npz',
                    X_train=data['train'][0], y_train=data['train'][1],
                    X_val=data['val'][0],     y_val=data['val'][1],
                    X_test=data['test'][0],   y_test=data['test'][1])
df[['rel_path', 'subject', 'closed', 'split']].to_csv(f'{PROC_DIR}/splits.csv', index=False)
json.dump(stats, open(f'{PROC_DIR}/normalization_stats.json', 'w'), indent=2)

for f in ['eye_dataset_64.npz', 'splits.csv', 'normalization_stats.json']:
    print(f'{f:28s} {os.path.getsize(os.path.join(PROC_DIR, f)) / 1e6:8.2f} MB')

In [ ]:
# Helper for Phase 2: load the processed data, already scaled to [0, 1]
def load_processed(path=f'{PROC_DIR}/eye_dataset_64.npz'):
    d = np.load(path)
    return {k: (d[k].astype(np.float32) / 255.0 if k.startswith('X') else d[k]) for k in d.files}

d = load_processed()
print({k: (v.shape, str(v.dtype), round(float(v.max()), 2)) for k, v in d.items()})

## Summary of preprocessing
* Grayscale + resize to 64x64 for every image.
* Pixels scaled to [0, 1] (mean/std from the training set stored for optional standardisation).
* Subject-wise 70/15/15 split with a leakage assertion.
* Class weights computed from training labels.
* Outputs are in `data/processed/`. Phase 2 can start from `load_processed()`.